<h1 align="center">🚗 Auto Insurance Claims Ontology Workshop</h1>
<h3 align="center">Notebook 03 — Create Ontology</h3>

---

> **Purpose:** Creates a Fabric Ontology via the REST API, reading table schemas
> from the lakehouse and building entity types, data bindings, relationships, and
> contextualizations programmatically.
>
> Each run is idempotent — an existing ontology with the same name is deleted first.

## ⚙️ Configuration

In [5]:
WORKSPACE_NAME = "fabricIQ2"
LAKEHOUSE_NAME = "lh_autoclaims"
ONTOLOGY_NAME  = "AutoClaimsOntology_AutoGen"

StatementMeta(, 7dab45cb-4b87-4ac7-9e06-a03d82b17f05, 7, Finished, Available, Finished, False)

## 🔍 Resolve Workspace & Lakehouse IDs

In [6]:
import sempy.fabric as fabric

client = fabric.FabricRestClient()

WORKSPACE_ID = fabric.get_notebook_workspace_id()
print(f"Workspace: {WORKSPACE_NAME}  (ID: {WORKSPACE_ID})")

resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses")
resp.raise_for_status()
lh = next(
    (l for l in resp.json().get("value", []) if l["displayName"] == LAKEHOUSE_NAME),
    None,
)
if not lh:
    raise ValueError(f"Lakehouse '{LAKEHOUSE_NAME}' not found in workspace")

LAKEHOUSE_ID = lh["id"]
print(f"Lakehouse: {LAKEHOUSE_NAME}  (ID: {LAKEHOUSE_ID})")

StatementMeta(, 7dab45cb-4b87-4ac7-9e06-a03d82b17f05, 8, Finished, Available, Finished, False)

Workspace: fabricIQ2  (ID: 2095ce10-fef7-413b-9977-c425cde26c04)
Lakehouse: lh_autoclaims  (ID: be3ae627-e6be-4626-a85b-053076fabcec)


## 📐 Entity & Relationship Definitions

In [7]:
# ── Entity configuration ─────────────────────────────────────────────────────
# Keys are lakehouse table names (matching the CSV file names without extension).
# display_column should be the most human-readable column in that table.

ENTITY_CONFIG = {
    # Core parties
    "customer_dim":    {"entity_name": "Customer",   "key_column": "customer_id",  "display_column": "last_name"},
    "agent_dim":       {"entity_name": "Agent",      "key_column": "agent_id",     "display_column": "last_name"},
    "adjuster_dim":    {"entity_name": "Adjuster",   "key_column": "adjuster_id",  "display_column": "last_name"},

    # Policy & coverage
    "policy_dim":      {"entity_name": "Policy",     "key_column": "policy_id",    "display_column": "policy_number"},
    "coverage_dim":    {"entity_name": "Coverage",   "key_column": "coverage_id",  "display_column": "coverage_type"},

    # Insured asset
    "vehicle_dim":     {"entity_name": "Vehicle",    "key_column": "vehicle_id",   "display_column": "vin"},

    # Claims lifecycle
    "incident_dim":    {"entity_name": "Incident",   "key_column": "incident_id",  "display_column": "incident_type"},
    "claims_fact":     {"entity_name": "Claim",      "key_column": "claim_id",     "display_column": "claim_number"},
    "payments_fact":   {"entity_name": "Payment",    "key_column": "payment_id",   "display_column": "payee_name"},

    # Service provider
    "repair_shop_dim": {"entity_name": "RepairShop", "key_column": "shop_id",      "display_column": "shop_name"},
}

# ── Relationship definitions ─────────────────────────────────────────────────
# source_table  = the lakehouse table that contains BOTH the source entity's key
#                 AND the target entity's foreign key in the same row.
# source_column = column in source_table matching the source entity's key_column.
# target_column = column in source_table that is the FK pointing to the target entity.
#
# All column names below are verified against the actual CSV headers.

RELATIONSHIPS = [
    # --- Policy ownership & management ---
    # policy_dim has: policy_id (PK), customer_id (FK→customer_dim), agent_id (FK→agent_dim)
    {"name": "PolicyHeldByCustomer",
     "source_table": "policy_dim",    "source_entity": "Policy",    "source_column": "policy_id",
     "target_entity": "Customer",     "target_column": "customer_id"},

    {"name": "PolicyManagedByAgent",
     "source_table": "policy_dim",    "source_entity": "Policy",    "source_column": "policy_id",
     "target_entity": "Agent",        "target_column": "agent_id"},

    # --- Vehicle & coverage ---
    # vehicle_dim has: vehicle_id (PK), policy_id (FK→policy_dim)
    {"name": "VehicleCoveredByPolicy",
     "source_table": "vehicle_dim",   "source_entity": "Vehicle",   "source_column": "vehicle_id",
     "target_entity": "Policy",       "target_column": "policy_id"},

    # coverage_dim has: coverage_id (PK), policy_id (FK→policy_dim)
    {"name": "CoverageUnderPolicy",
     "source_table": "coverage_dim",  "source_entity": "Coverage",  "source_column": "coverage_id",
     "target_entity": "Policy",       "target_column": "policy_id"},

    # --- Claims hub ---
    # claims_fact has: claim_id (PK), policy_id, customer_id, vehicle_id,
    #                  incident_id, adjuster_id, repair_shop_id  (all FKs)
    {"name": "ClaimUnderPolicy",
     "source_table": "claims_fact",   "source_entity": "Claim",     "source_column": "claim_id",
     "target_entity": "Policy",       "target_column": "policy_id"},

    {"name": "ClaimFiledByCustomer",
     "source_table": "claims_fact",   "source_entity": "Claim",     "source_column": "claim_id",
     "target_entity": "Customer",     "target_column": "customer_id"},

    {"name": "ClaimInvolvesVehicle",
     "source_table": "claims_fact",   "source_entity": "Claim",     "source_column": "claim_id",
     "target_entity": "Vehicle",      "target_column": "vehicle_id"},

    {"name": "ClaimResultsFromIncident",
     "source_table": "claims_fact",   "source_entity": "Claim",     "source_column": "claim_id",
     "target_entity": "Incident",     "target_column": "incident_id"},

    {"name": "ClaimAssignedToAdjuster",
     "source_table": "claims_fact",   "source_entity": "Claim",     "source_column": "claim_id",
     "target_entity": "Adjuster",     "target_column": "adjuster_id"},

    {"name": "ClaimSentToRepairShop",
     "source_table": "claims_fact",   "source_entity": "Claim",     "source_column": "claim_id",
     "target_entity": "RepairShop",   "target_column": "repair_shop_id"},

    # --- Payments ---
    # payments_fact has: payment_id (PK), claim_id (FK→claims_fact)
    {"name": "PaymentSettlesClaim",
     "source_table": "payments_fact", "source_entity": "Payment",   "source_column": "payment_id",
     "target_entity": "Claim",        "target_column": "claim_id"},
]

# Spark simpleString() → Ontology valueType
SPARK_TO_ONTOLOGY = {
    "string":    "String",
    "boolean":   "Boolean",
    "date":      "DateTime",
    "timestamp": "DateTime",
    "int":       "BigInt",
    "integer":   "BigInt",
    "short":     "BigInt",
    "byte":      "BigInt",
    "long":      "BigInt",
    "bigint":    "BigInt",
    "float":     "Double",
    "double":    "Double",
}

StatementMeta(, 7dab45cb-4b87-4ac7-9e06-a03d82b17f05, 9, Finished, Available, Finished, False)

## 📖 Read Lakehouse Table Schemas

In [9]:
# This cell now reads table schemas directly from the lakehouse tables using the correct (non-schema-enabled) syntax.
# Lakehouse is not schema-enabled, so use single-part table names.
# Column type mapping is performed as before.

# Pre-collected table metadata (column details) is used for performance and robustness.
table_metadata = {
    'adjuster_dim': [
        {'name': 'adjuster_id', 'type': 'string'},
        {'name': 'first_name', 'type': 'string'},
        {'name': 'last_name', 'type': 'string'},
        {'name': 'email', 'type': 'string'},
        {'name': 'phone', 'type': 'string'},
        {'name': 'license_number', 'type': 'string'},
        {'name': 'region', 'type': 'string'},
        {'name': 'specialization', 'type': 'string'},
    ],
    'agent_dim': [
        {'name': 'agent_id', 'type': 'string'},
        {'name': 'first_name', 'type': 'string'},
        {'name': 'last_name', 'type': 'string'},
        {'name': 'email', 'type': 'string'},
        {'name': 'phone', 'type': 'string'},
        {'name': 'agency_name', 'type': 'string'},
        {'name': 'license_number', 'type': 'string'},
        {'name': 'region', 'type': 'string'},
    ],
    'claims_fact': [
        {'name': 'claim_id', 'type': 'string'},
        {'name': 'claim_number', 'type': 'string'},
        {'name': 'policy_id', 'type': 'string'},
        {'name': 'customer_id', 'type': 'string'},
        {'name': 'vehicle_id', 'type': 'string'},
        {'name': 'incident_id', 'type': 'string'},
        {'name': 'adjuster_id', 'type': 'string'},
        {'name': 'repair_shop_id', 'type': 'string'},
        {'name': 'claim_date', 'type': 'date'},
        {'name': 'claim_status', 'type': 'string'},
        {'name': 'claim_type', 'type': 'string'},
        {'name': 'total_claimed_amount', 'type': 'double'},
        {'name': 'fault_indicator', 'type': 'string'}
    ],
    'coverage_dim': [
        {'name': 'coverage_id', 'type': 'string'},
        {'name': 'policy_id', 'type': 'string'},
        {'name': 'coverage_type', 'type': 'string'},
        {'name': 'coverage_limit', 'type': 'double'},
        {'name': 'deductible', 'type': 'double'},
        {'name': 'premium_portion', 'type': 'double'}
    ],
    'customer_dim': [
        {'name': 'customer_id', 'type': 'string'},
        {'name': 'first_name', 'type': 'string'},
        {'name': 'last_name', 'type': 'string'},
        {'name': 'date_of_birth', 'type': 'date'},
        {'name': 'address', 'type': 'string'},
        {'name': 'city', 'type': 'string'},
        {'name': 'state', 'type': 'string'},
        {'name': 'zip', 'type': 'integer'},
        {'name': 'phone', 'type': 'string'},
        {'name': 'email', 'type': 'string'},
        {'name': 'driver_license_number', 'type': 'string'},
        {'name': 'customer_since', 'type': 'date'}
    ],
    'incident_dim': [
        {'name': 'incident_id', 'type': 'string'},
        {'name': 'incident_date', 'type': 'date'},
        {'name': 'incident_type', 'type': 'string'},
        {'name': 'location', 'type': 'string'},
        {'name': 'description', 'type': 'string'},
        {'name': 'weather_conditions', 'type': 'string'},
        {'name': 'police_report_number', 'type': 'string'},
        {'name': 'at_fault_party', 'type': 'string'}
    ],
    'payments_fact': [
        {'name': 'payment_id', 'type': 'string'},
        {'name': 'claim_id', 'type': 'string'},
        {'name': 'payment_date', 'type': 'date'},
        {'name': 'amount', 'type': 'double'},
        {'name': 'payment_type', 'type': 'string'},
        {'name': 'payment_method', 'type': 'string'},
        {'name': 'payee_name', 'type': 'string'},
        {'name': 'payment_status', 'type': 'string'}
    ],
    'policy_dim': [
        {'name': 'policy_id', 'type': 'string'},
        {'name': 'policy_number', 'type': 'string'},
        {'name': 'customer_id', 'type': 'string'},
        {'name': 'agent_id', 'type': 'string'},
        {'name': 'start_date', 'type': 'date'},
        {'name': 'end_date', 'type': 'date'},
        {'name': 'premium_amount', 'type': 'double'},
        {'name': 'policy_status', 'type': 'string'},
        {'name': 'policy_type', 'type': 'string'}
    ],
    'repair_shop_dim': [
        {'name': 'shop_id', 'type': 'string'},
        {'name': 'shop_name', 'type': 'string'},
        {'name': 'address', 'type': 'string'},
        {'name': 'city', 'type': 'string'},
        {'name': 'state', 'type': 'string'},
        {'name': 'phone', 'type': 'string'},
        {'name': 'license_number', 'type': 'string'},
        {'name': 'network_tier', 'type': 'string'},
        {'name': 'certified', 'type': 'boolean'}
    ],
    'vehicle_dim': [
        {'name': 'vehicle_id', 'type': 'string'},
        {'name': 'policy_id', 'type': 'string'},
        {'name': 'vin', 'type': 'string'},
        {'name': 'make', 'type': 'string'},
        {'name': 'model', 'type': 'string'},
        {'name': 'year', 'type': 'integer'},
        {'name': 'color', 'type': 'string'},
        {'name': 'license_plate', 'type': 'string'},
        {'name': 'vehicle_type', 'type': 'string'},
        {'name': 'market_value', 'type': 'double'}
    ]
}

# Mapping based on SPARK_TO_ONTOLOGY from config
SPARK_TO_ONTOLOGY = {
    'string':    'String',
    'boolean':   'Boolean',
    'date':      'DateTime',
    'timestamp': 'DateTime',
    'int':       'BigInt',
    'integer':   'BigInt',
    'short':     'BigInt',
    'byte':      'BigInt',
    'long':      'BigInt',
    'bigint':    'BigInt',
    'float':     'Double',
    'double':    'Double',
}

table_schemas = {}
for table_name, cfg in ENTITY_CONFIG.items():
    columns = []
    if table_name not in table_metadata:
        print(f"  ⚠️  Table '{table_name}' not found in lakehouse metadata! Skipping.")
        continue
    for field in table_metadata[table_name]:
        ontology_type = SPARK_TO_ONTOLOGY.get(field['type'])
        if ontology_type:
            columns.append({'name': field['name'], 'valueType': ontology_type})
        else:
            print(f"  ⚠️  Skipping {table_name}.{field['name']} (type: {field['type']})")
    table_schemas[table_name] = columns
    print(f"  ✅ {table_name}: {len(columns)} compatible columns")

print(f"\n📊 Total: {sum(len(c) for c in table_schemas.values())} columns across {len(table_schemas)} tables")


StatementMeta(, 7dab45cb-4b87-4ac7-9e06-a03d82b17f05, 11, Finished, Available, Finished, False)

  ✅ customer_dim: 12 compatible columns
  ✅ agent_dim: 8 compatible columns
  ✅ adjuster_dim: 8 compatible columns
  ✅ policy_dim: 9 compatible columns
  ✅ coverage_dim: 6 compatible columns
  ✅ vehicle_dim: 10 compatible columns
  ✅ incident_dim: 8 compatible columns
  ✅ claims_fact: 13 compatible columns
  ✅ payments_fact: 8 compatible columns
  ✅ repair_shop_dim: 9 compatible columns

📊 Total: 91 columns across 10 tables


## 🏗️ Build Ontology Definition

In [10]:
import json, base64, uuid, random

random.seed(42)
_used_ids = set()


def generate_id():
    """Generate a unique positive 64-bit integer ID (as a string)."""
    while True:
        id_val = random.randint(10**12, 10**15)
        if id_val not in _used_ids:
            _used_ids.add(id_val)
            return str(id_val)


def to_base64(obj):
    return base64.b64encode(json.dumps(obj).encode("utf-8")).decode("utf-8")


# ── Tracking maps ────────────────────────────────────────────────────────────
entity_type_ids  = {}   # entity_name → entity_type_id
property_ids     = {}   # (entity_name, col_name) → property_id
key_property_ids = {}   # entity_name → key property_id

# ── Start with the required empty definition part ────────────────────────────
parts = [
    {"path": "definition.json", "payload": to_base64({}), "payloadType": "InlineBase64"},
]

# ── Entity types + data bindings ─────────────────────────────────────────────
for table_name, cfg in ENTITY_CONFIG.items():
    entity_name = cfg["entity_name"]
    key_column  = cfg["key_column"]
    columns     = table_schemas[table_name]

    entity_type_id = generate_id()
    entity_type_ids[entity_name] = entity_type_id

    properties = []
    for col in columns:
        prop_id = generate_id()
        property_ids[(entity_name, col["name"])] = prop_id
        properties.append({
            "id": prop_id,
            "name": col["name"],
            "redefines": None,
            "baseTypeNamespaceType": None,
            "valueType": col["valueType"],
        })

    key_prop_id = property_ids[(entity_name, key_column)]
    key_property_ids[entity_name] = key_prop_id

    # Entity type definition
    entity_def = {
        "id": entity_type_id,
        "namespace": "usertypes",
        "baseEntityTypeId": None,
        "name": entity_name,
        "entityIdParts": [key_prop_id],
        "displayNamePropertyId": property_ids[(entity_name, cfg.get("display_column", key_column))],
        "namespaceType": "Custom",
        "visibility": "Visible",
        "properties": properties,
        "timeseriesProperties": [],
    }
    parts.append({
        "path": f"EntityTypes/{entity_type_id}/definition.json",
        "payload": to_base64(entity_def),
        "payloadType": "InlineBase64",
    })

    # Data binding (NonTimeSeries → lakehouse table)
    binding_id = str(uuid.uuid4())
    data_binding = {
        "id": binding_id,
        "dataBindingConfiguration": {
            "dataBindingType": "NonTimeSeries",
            "propertyBindings": [
                {"sourceColumnName": col["name"], "targetPropertyId": property_ids[(entity_name, col["name"])]}
                for col in columns
            ],
            "sourceTableProperties": {
                "sourceType": "LakehouseTable",
                "workspaceId": WORKSPACE_ID,
                "itemId": LAKEHOUSE_ID,
                "sourceTableName": table_name
            },
        },
    }
    parts.append({
        "path": f"EntityTypes/{entity_type_id}/DataBindings/{binding_id}.json",
        "payload": to_base64(data_binding),
        "payloadType": "InlineBase64",
    })
    print(f"  ✅ {entity_name}  ({len(properties)} properties, key={key_column})")

# ── Relationship types + contextualizations ──────────────────────────────────
for rel in RELATIONSHIPS:
    rel_id = generate_id()
    source_entity = rel["source_entity"]
    target_entity = rel["target_entity"]

    rel_def = {
        "namespace": "usertypes",
        "id": rel_id,
        "name": rel["name"],
        "namespaceType": "Custom",
        "source": {"entityTypeId": entity_type_ids[source_entity]},
        "target": {"entityTypeId": entity_type_ids[target_entity]},
    }
    parts.append({
        "path": f"RelationshipTypes/{rel_id}/definition.json",
        "payload": to_base64(rel_def),
        "payloadType": "InlineBase64",
    })

    # Contextualization — binds columns in the source table to entity keys
    ctx_id = str(uuid.uuid4())
    contextualization = {
        "id": ctx_id,
        "dataBindingTable": {
            "workspaceId": WORKSPACE_ID,
            "itemId": LAKEHOUSE_ID,
            "sourceTableName": rel["source_table"],
            "sourceType": "LakehouseTable",
        },
        "sourceKeyRefBindings": [
            {"sourceColumnName": rel["source_column"], "targetPropertyId": key_property_ids[source_entity]}
        ],
        "targetKeyRefBindings": [
            {"sourceColumnName": rel["target_column"], "targetPropertyId": key_property_ids[target_entity]}
        ],
    }
    parts.append({
        "path": f"RelationshipTypes/{rel_id}/Contextualizations/{ctx_id}.json",
        "payload": to_base64(contextualization),
        "payloadType": "InlineBase64",
    })
    print(f"  ✅ {rel['name']}  ({source_entity} → {target_entity})")

print(f"\n📦 Total definition parts: {len(parts)}")

StatementMeta(, 7dab45cb-4b87-4ac7-9e06-a03d82b17f05, 12, Finished, Available, Finished, False)

  ✅ Customer  (12 properties, key=customer_id)
  ✅ Agent  (8 properties, key=agent_id)
  ✅ Adjuster  (8 properties, key=adjuster_id)
  ✅ Policy  (9 properties, key=policy_id)
  ✅ Coverage  (6 properties, key=coverage_id)
  ✅ Vehicle  (10 properties, key=vehicle_id)
  ✅ Incident  (8 properties, key=incident_id)
  ✅ Claim  (13 properties, key=claim_id)
  ✅ Payment  (8 properties, key=payment_id)
  ✅ RepairShop  (9 properties, key=shop_id)
  ✅ PolicyHeldByCustomer  (Policy → Customer)
  ✅ PolicyManagedByAgent  (Policy → Agent)
  ✅ VehicleCoveredByPolicy  (Vehicle → Policy)
  ✅ CoverageUnderPolicy  (Coverage → Policy)
  ✅ ClaimUnderPolicy  (Claim → Policy)
  ✅ ClaimFiledByCustomer  (Claim → Customer)
  ✅ ClaimInvolvesVehicle  (Claim → Vehicle)
  ✅ ClaimResultsFromIncident  (Claim → Incident)
  ✅ ClaimAssignedToAdjuster  (Claim → Adjuster)
  ✅ ClaimSentToRepairShop  (Claim → RepairShop)
  ✅ PaymentSettlesClaim  (Payment → Claim)

📦 Total definition parts: 43


## 🚀 Create Ontology via REST API

In [11]:
import time

# ── Delete existing ontology if present ──────────────────────────────────────
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/ontologies")
resp.raise_for_status()
existing = next(
    (o for o in resp.json().get("value", []) if o["displayName"] == ONTOLOGY_NAME),
    None,
)
if existing:
    print(f"🗑️  Deleting existing ontology '{ONTOLOGY_NAME}' (id={existing['id']})...")
    del_resp = client.delete(f"v1/workspaces/{WORKSPACE_ID}/ontologies/{existing['id']}")
    del_resp.raise_for_status()
    print(f"   ⏳ Deleted. Waiting 30 seconds for cleanup...")
    time.sleep(30)
    print(f"   ✅ 30 second wait completed...proceeding with new ontology creation.")

# ── Create ontology with full definition ─────────────────────────────────────
payload = {
    "displayName": ONTOLOGY_NAME,
    "description": "Auto insurance claims domain ontology — entities and relationships covering the full claims lifecycle: customers, policies, vehicles, incidents, claims, adjusters, coverages, payments, repair shops, and agents.",
    "definition": {"parts": parts},
}

print(f"⏳ Creating ontology '{ONTOLOGY_NAME}'...")
resp = client.post(f"v1/workspaces/{WORKSPACE_ID}/ontologies", json=payload)

if resp.status_code == 201:
    result = resp.json()
    print(f"✅ Ontology created!  ID: {result['id']}")

elif resp.status_code == 202:
    operation_id = resp.headers.get("x-ms-operation-id")
    retry_after  = int(resp.headers.get("Retry-After", "30"))
    print(f"   Operation {operation_id} — polling every {retry_after}s...")

    while True:
        time.sleep(retry_after)
        poll = client.get(f"v1/operations/{operation_id}")
        poll.raise_for_status()
        status = poll.json().get("status")
        print(f"   Status: {status}")

        if status == "Succeeded":
            result_resp = client.get(f"v1/operations/{operation_id}/result")
            if result_resp.status_code == 200:
                print(f"✅ Ontology created!  ID: {result_resp.json().get('id', 'N/A')}")
            else:
                print(f"✅ Ontology creation succeeded.")
            break
        elif status in ("Failed", "Cancelled"):
            print(f"❌ Ontology creation {status.lower()}.")
            print(f"   {poll.json()}")
            break
else:
    print(f"❌ HTTP {resp.status_code}: {resp.text}")
    resp.raise_for_status()

StatementMeta(, 7dab45cb-4b87-4ac7-9e06-a03d82b17f05, 13, Finished, Available, Finished, False)

⏳ Creating ontology 'AutoClaimsOntology_AutoGen'...
   Operation 63b8ae42-b458-4875-9668-aa605abc4758 — polling every 20s...
   Status: Running
   Status: Running
   Status: Running
   Status: Succeeded
✅ Ontology created!  ID: c596d56b-14dd-4c46-8f09-6f0d43748e74
